# Chapter 5 — Milvus with Movies Dataset

## 1. Explore the Dataset

In [2]:
import itertools

import polars as pl
from datasets import load_dataset
from pymilvus import DataType, MilvusClient

DS = "MongoDB/embedded_movies"

# Stream the dataset to avoid downloading it all at once
movies_ds = load_dataset(DS, streaming=True, split="train")
print(movies_ds.features)

README.md: 0.00B [00:00, ?B/s]

{'plot': Value('string'), 'runtime': Value('int64'), 'genres': List(Value('string')), 'fullplot': Value('string'), 'directors': List(Value('string')), 'writers': List(Value('string')), 'countries': List(Value('string')), 'poster': Value('string'), 'languages': List(Value('string')), 'cast': List(Value('string')), 'title': Value('string'), 'num_mflix_comments': Value('int64'), 'rated': Value('string'), 'imdb': {'id': Value('int64'), 'rating': Value('float64'), 'votes': Value('int64')}, 'awards': {'nominations': Value('int64'), 'text': Value('string'), 'wins': Value('int64')}, 'type': Value('string'), 'metacritic': Value('int64'), 'plot_embedding': List(Value('float64'))}


## 2. Load 4 Drama movies from the dataset

In [10]:
drama_rows = (
    row for row in movies_ds
    if row.get("plot_embedding") and row.get("genres") and "Drama" in row["genres"]
)

rows = list(itertools.islice(drama_rows, 4))

# Preview: show all genres to see how Drama co-occurs with other genres
preview = pl.DataFrame([
    {"id": i, "title": row["title"], "genres": row["genres"]}
    for i, row in enumerate(rows)
])
preview

id,title,genre
i64,str,list[str]
0,"""Beau Geste""","[""Action"", ""Adventure"", ""Drama""]"
1,"""Men Without Women""","[""Action"", ""Drama""]"
2,"""The Crowd Roars""","[""Drama"", ""Action""]"
3,"""Scarface""","[""Action"", ""Crime"", ""Drama""]"


## 3. Define Schema & Create Collection

`genre` is used as the partition key so Milvus automatically routes each document to a logical partition.

In [5]:
client = MilvusClient(uri="http://localhost:19530")

schema = MilvusClient.create_schema(auto_id=False, enable_dynamic_field=False)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="title", datatype=DataType.VARCHAR, max_length=512)
schema.add_field(field_name="plot", datatype=DataType.VARCHAR, max_length=4096)
schema.add_field(field_name="genre", datatype=DataType.VARCHAR, max_length=64, is_partition_key=True)
schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=1536)

# create collection with partition key field "genre"
client.create_collection(
    collection_name="coll_enables_partitionkey",
    schema=schema,
)

# create collection with 4 shards
client.create_collection(
    collection_name="coll_enables_four_shards",
    schema=schema,
    num_shards=4,
)

# create collection with 2 shards and 2 partitions with partition key
client.create_collection(
    collection_name="coll_2_shards_2_partitions",
    schema=schema,
    num_shards=2,
    num_partitions=2,
)

## 4. Insert Drama Sample

Connect to a deployed Milvus instance for this step.

In [16]:
drama_entities = [
    {
        "id":        i,
        "title":     row["title"],
        "plot":      row.get("plot", ""),
        "genre":     "Drama",
        "embedding": row["plot_embedding"],
    }
    for i, row in enumerate(rows)
]

for entity in drama_entities:
    print(f"Entity {entity['id']}: genre={entity['genre']} (partition key)")

client.insert(collection_name="coll_2_shards_2_partitions", data=drama_entities)

Entity 0: genre=Action (partition key)
Entity 1: genre=Action (partition key)
Entity 2: genre=Drama (partition key)
Entity 3: genre=Action (partition key)


{'insert_count': 4, 'ids': [0, 1, 2, 3], 'cost': 0}